In [ ]:
import sympy as sp
import numpy as np
from scipy.special import jv
from scipy.sparse import lil_matrix
import matplotlib.pyplot as plt
from scipy.linalg import block_diag
from scipy.optimize import root
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting
from Plot_setting import *

# Hill method — period-1 orbit (corrected 08/10/2026)

Copy of `Hill_method.ipynb` with these changes:

1. **Floquet exponents**: `Floq_coef` keeps the **2n = 4** genuine exponents (Detroux et al. 2015), not 4n.
   They are selected by the centring of their eigenvector in harmonic space, keeping one member per family
   $\lambda + i m$. This stays well defined at period doubling, where Moore's $|\mathrm{Im}\,\lambda|$
   criterion is ambiguous. Plots use all 2n exponents (`[:, i]` instead of `[:, 2*i]`).
   Check: $\sum \mathrm{Re}\,\lambda = -\mathrm{tr}(M^{-1}C)$ (Liouville), printed after the sweep.
2. **Damping**: same convention as the Julia simulations (`SystemATS`: $\ddot\phi_b = \dots - 2\kappa_b\dot\phi_b$),
   so $C = \mathrm{diag}(0, \kappa_b')$ with $\kappa_b' = \kappa_b/\omega_b$. Previously it was $\kappa_b'/2$.
3. **Parameters**: $\kappa_b = \pi/10.4$, as in `lyapunov_heatmap.ipynb`. Previously $1/10.4$, with an extra factor $\pi$
   in the first parameter cell only.

`db_dz` is vectorised; its result is unchanged. Figures are saved under new names (suffix `_08_10_2026`).

**12/10/2026: definition of $g_2$.** The pump is static ($\omega_p = 0$) and enters the EOMs as $\sin\varepsilon_p$, so
$g_2 = E_J\sin(\varepsilon_p)\,\varphi_a^2\varphi_b$ (`g2_of`), as in the article. It was $J_1(\varepsilon_p)E_J\varphi_a^2\varphi_b$
(oscillating pump, Leon's paper), about half as large. The drive is still $\varepsilon_d = (\varepsilon_d/g_2)\,g_2$, so at a given ratio
$\varepsilon_d$ is now about twice as large and $\varepsilon_d/2g_2 = |\alpha|^2$ at lowest order. The hard-coded seeds were converged with the
former definition (`g2_J1_of`): they are first continued in $\varepsilon_d$ up to the new value. Outputs have the suffix `_12_10_2026`.


**19/10/2026: why the sweep fails at high $\varepsilon_p$.** At $\varepsilon_d = 10.12\,g_2$ the cat orbit ends at a fold (saddle-node of limit cycles), at $\varepsilon_p^\ast \simeq 0.786$ ($g_2/2\pi \simeq 97$ MHz).
Above it no cat solution exists, so Powell fails or jumps to another branch. The sweep below stops at 0.785, just short of it.
Up to the fold the multipliers stay at $|\mu| \le 0.973$. A real multiplier crosses $+1$ only at the fold itself.
See `Hill_method_cat_fold.ipynb`: pseudo-arclength continuation through the fold, a monodromy cross-check, the fold line vs $\varepsilon_d/g_2$ on the cat-start map, and the basin scan.

Hamiltonian $$H = \omega_a(x_a^2/2+p_a^2/2) + \omega_b(x_b^2/2+p_b^2/2) + \varepsilon_d\cos(\omega_d t)\sqrt 2 x_b-2E_J\sin(\varepsilon_p)[\sin(\sqrt 2 \varphi_ax_a+ \sqrt 2 \varphi_bx_b)-\sqrt 2 \varphi_ax_a- \sqrt 2 \varphi_bx_b]$$

EOMs
\begin{equation}
\begin{cases}
\dot x_a = \omega_a p_a \\
\dot p_a = -\omega_a x_a +2E_J\sin(\varepsilon_p)\sqrt 2 \varphi_a[\cos(\sqrt 2 \varphi_ax_a+ \sqrt 2 \varphi_bx_b)-1] \\
\dot x_b = \omega_b p_b \\
\dot p_b = -\omega_b x_b - \sqrt 2 \varepsilon_d\cos(\omega_d t)+2E_J\sin(\varepsilon_p)\sqrt 2 \varphi_b[\cos(\sqrt 2 \varphi_ax_a+ \sqrt 2 \varphi_bx_b)-1] -2\kappa_b p_b\\
\end{cases}
\end{equation}

We rescale time and the variables
\begin{equation}
\begin{cases}
 dx_a^{\lambda}/d\tau = p_a^{\lambda} \\
dp_a^{\lambda}/d\tau = - x_a^{\lambda} +2E_J\sin(\varepsilon_p)\sqrt 2 \varphi_a^2 [\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1]/\omega_a \\
dx_b^{\lambda}/d(2\tau) = p_b^{\lambda} \\
dp_b^{\lambda}/d(2\tau) = -x_b^{\lambda} - \sqrt 2 \varepsilon_d\varphi_b\cos(2\tau)/\omega_b+2E_J\sin(\varepsilon_p)\sqrt 2 \varphi_b^2 [\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1]/\omega_b-2\kappa_b p_b^{\lambda}/\omega_b\\
\end{cases}
\end{equation}

\begin{equation}
\begin{cases}
 dx_a^{\lambda}/d\tau = p_a^{\lambda} \\
dp_a^{\lambda}/d\tau = - x_a^{\lambda} +\sqrt 2\beta_a' [\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1] \\
dx_b^{\lambda}/d(2\tau) = p_b^{\lambda} \\
dp_b^{\lambda}/d(2\tau) = -x_b^{\lambda} - \sqrt 2 \varepsilon_d'\cos(2\tau)+\sqrt 2\beta_b'[\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1]-2\kappa_b' p_b^{\lambda}\\
\end{cases}
\end{equation}

Finally we write the second order equations on the positions

\begin{equation}
\begin{cases}
\frac{d^2x_a^{\lambda}}{d\tau^2} = - x_a^{\lambda} +\sqrt 2\beta_a' [\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1] \\
\frac{1}{4}\frac{d^2x_b^{\lambda}}{d\tau^2} = -x_b^{\lambda} - \sqrt 2 \varepsilon_d'\cos(2\tau)+\sqrt 2\beta_b'[\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1] -\kappa_b' dx_b^{\lambda}/d\tau\\
\end{cases}
\end{equation}

We introduce $\bold X = (x_a^{\lambda}, x_b^{\lambda})^T$ and we get

\begin{equation}
\begin{split}
\begin{pmatrix}1&0\\0&1/4\end{pmatrix}\ddot{\bold X}+\begin{pmatrix}0&0\\0&\kappa_b'\end{pmatrix}\dot{\bold X}+\begin{pmatrix}1&0\\0&1\end{pmatrix}\bold X-\begin{pmatrix}\beta_a'\\\beta_b'\end{pmatrix}\sqrt 2[\cos(\sqrt 2 x_a^{\lambda}+ \sqrt 2 x_b^{\lambda})-1] =& -\begin{pmatrix}0\\1\end{pmatrix}\sqrt 2 \varepsilon_d'\cos(2\tau)\\
M\ddot{\bold X} + C\dot{\bold X} + K\bold X + \bold f_{nl}(\bold X)= \bold f_{ext}(\omega = 2,\tau)
\end{split}
\end{equation}

Then we are going to write the position as fFourier series
$$\bold X(t)=\frac{\bold c_0^x}{\sqrt 2}+\sum_{k=1}^{N_H}\biggl(\bold s_k^x\sin(kt)+\bold c_k^x\cos(kt)\biggr)$$

Let $$\bold f_{nl}(\bold X) = \frac{\bold c_0^f}{\sqrt 2}+\sum_{k=1}^{N_H}\biggl(\bold s_k^f\sin(kt)+\bold c_k^f\cos(kt)\biggr)$$

Let $z = [\bold c_0^x, \bold s_1^x, \bold c_1^x, ...]$ and $b = [\bold c_0^f, \bold s_1^f, \bold c_1^f, ...]$

In [ ]:

h_bar=1
e=1
phi_quantum=1/2

#### Caracterization of the circuit #####

w_a = 25.338776456203686
w_b = 2*w_a

phi_a = 0.11
Z_a = 2*phi_quantum**2*phi_a**2
C_a_inv = Z_a*w_a
E_Ca = C_a_inv/2

phi_b = 0.204
Z_b = 2*phi_quantum**2*phi_b**2
C_b_inv = Z_b*w_b
E_Cb = C_b_inv/2

E_J = 37.12*2*np.pi*1.5

#Environment
kappa_b=np.pi/10.4; # 1/ns, same value and convention as the Julia notebooks (phi_b'' = ... - 2 kappa_b phi_b')

w_d=w_b # Drive frequency
w_p=2*w_a-w_d # Pump Frequency

epsilon_p = 0.58
def g2_of(eps_p):
    """Two-photon rate for the static pump (w_p = 0), as in the EOMs: g2 = E_J sin(eps_p) phi_a^2 phi_b."""
    return np.sin(eps_p)*E_J*phi_a**2*phi_b

def g2_J1_of(eps_p):
    """Former definition (oscillating pump, Leon's paper). Only used to continue the seeds below, which were converged with it."""
    return jv(1, eps_p)*E_J*phi_a**2*phi_b

g2 = g2_of(epsilon_p)
epsilon_d = 8.12*g2#0.75*g2_max

beta_a_p, beta_b_p = 2*E_J*np.sin(epsilon_p)*phi_a**2/(w_a), 2*E_J*np.sin(epsilon_p)*phi_b**2/(w_b)
B = np.array([beta_a_p, beta_b_p])  # shape (2,)
eps_d_p = epsilon_d*phi_b/w_b
kappa_b_p = kappa_b / w_b

z_0_init = np.array([-2.09501523e-02, -3.60273363e-02,  3.26246220e-01, -3.28386996e-17,
       -6.78344780e-03,  1.43507376e-17, -1.72514305e-04, -3.13979476e-03,
       -4.23320965e-03, -7.46552651e-02, -8.01435361e-04, -8.82665625e-03,
        5.05426053e-05,  4.29993690e-04,  7.79954440e-06,  6.01698525e-05,
        1.00817384e-04,  8.67341068e-04])



In [ ]:
def build_L(M, C, K, N_H, omega=1.0, nu=1.0):
    """
    Build linear HB operator L for ordering:
    [c0, s1, c1, s2, c2, ..., s_NH, c_NH] with each block in R^n.
    Returns L of shape (n*(2N_H+1), n*(2N_H+1)).
    """
    n = M.shape[0]
    dim = n * (2*N_H + 1)
    L = np.zeros((dim, dim), dtype=float)

    # helper to locate block indices in z
    def idx_c0():      return 0
    def idx_sk(k):     return 1 + 2*(k-1)
    def idx_ck(k):     return 2 + 2*(k-1)

    # DC: (K) acting on c0 (with your 1/sqrt2 convention, L is still K here
    # because the same scaling appears in Gamma and Gamma^+; keep L=K)
    i0 = idx_c0()
    L[i0*n:(i0+1)*n, i0*n:(i0+1)*n] = K

    # Harmonics
    for k in range(1, N_H+1):
        A = K - ((k*omega/nu)**2)*M  # n×n
        B = (k*omega/nu) * C          # n×n

        is_ = idx_sk(k)
        ic_ = idx_ck(k)

        # sine row blocks
        L[is_*n:(is_+1)*n, is_*n:(is_+1)*n] = A
        L[is_*n:(is_+1)*n, ic_*n:(ic_+1)*n] = -B

        # cosine row blocks
        L[ic_*n:(ic_+1)*n, is_*n:(is_+1)*n] = B
        L[ic_*n:(ic_+1)*n, ic_*n:(ic_+1)*n] = A

    return L

def build_b_ext(N_H, eps_d_p, nu = 2):
    """
    b_ext in the same ordering as z:
    [c0, s1, c1, s2, c2, ..., s_NH, c_NH], each block in R^2.
    Forcing: f_ext = -sqrt(2)*eps_d' * cos(2 tau) * e2.
    """
    n = 2
    b = np.zeros(n*(2*N_H+1), dtype=float)

    def idx_ck(k): return 2 + 2*(k-1)

    if N_H >= nu:
        ic2 = idx_ck(nu)
        # e2 = [0,1]
        b[ic2*n:(ic2+1)*n] = np.array([0.0, -np.sqrt(2.0)*eps_d_p])

    return b

In [ ]:
def build_Gamma(t, N_H, n, omega=2.0, nu=2.0):
    """
    Build Gamma = I_n ⊗ Phi with omega/nu = 1.

    t   : array of shape (N,)
    N_H : number of harmonics
    n   : number of DOFs
    """
    t = np.asarray(t)
    N = t.size
    k = np.arange(1, N_H + 1)

    # N x N_H
    S = np.sin(np.outer(t, k*omega/nu))
    C = np.cos(np.outer(t, k*omega/nu))

    # Build Phi: N x (2*N_H + 1) with columns [1/sqrt2, sin(1t), cos(1t), sin(2t), cos(2t), ...]
    Phi = np.empty((N, 2 * N_H + 1), dtype=float)
    Phi[:, 0] = 1.0 / np.sqrt(2.0)
    Phi[:, 1::2] = S
    Phi[:, 2::2] = C

    # Gamma: (n*N) x (n*(2*N_H+1))
    Gamma = np.kron(Phi, np.eye(n))
    return Gamma

def x_tilde_to_X(tildex, n, N):
    # tildex is time-major: [x(t1); x(t2); ...], each x(tj) is length n
    return tildex.reshape(N, n).T   # -> (n, N)

def X_to_x_tilde(X):
    # X is (n, N); return time-major vector
    return X.T.reshape(-1)

def f_nl_time(X):
    """
    X: shape (n, N) with n=2
    returns Fnl: shape (n, N)
    """
    y = np.sqrt(2.0) * (X[0, :] + X[1, :])          # shape (N,)
    g = np.cos(y) - 1.0                             # shape (N,)
    Fnl = -np.sqrt(2.0) * B[:, None] * g[None, :]    # (2,N)
    return Fnl

def b_nl(z, Gamma, Gamma_pinv, n, N):
    x_tilde = Gamma @ z
    X = x_tilde_to_X(x_tilde, n, N)
    Fnl = f_nl_time(X)              # (n,N)
    f_tilde = X_to_x_tilde(Fnl)     # (nN,)
    return Gamma_pinv @ f_tilde     # (n*(2N_H+1),)

In [ ]:
def residual(z, L, b_ext, Gamma, Gamma_pinv, n, N):
    return L @ z + b_nl(z, Gamma, Gamma_pinv, n, N) - b_ext

def dfnl_dX_blocks(X):
    """
    returns list of N blocks J_j = ∂f_nl/∂X at each time sample
    each block is 2x2
    """
    y = np.sqrt(2.0) * (X[0, :] + X[1, :])
    s = np.sin(y)  # shape (N,)

    # J_j = 2 sin(y_j) * [[beta_a, beta_a],[beta_b, beta_b]]
    base = np.array([[beta_a_p, beta_a_p],
                     [beta_b_p, beta_b_p]])  # 2x2

    J_blocks = [(2.0 * s[j]) * base for j in range(X.shape[1])]
    return J_blocks

def build_dftilde_dx_tilde(X, n, N):
    """
    Block diagonal (nN x nN) with n×n blocks per time sample,
    consistent with Gamma = kron(Phi, I_n) (time-major stacking):
      tilde x = [x(t1); x(t2); ...], each x(tj) in R^n.
    """
    J_blocks = dfnl_dX_blocks(X)  # length N, each (n,n)
    Jbig = lil_matrix((n*N, n*N), dtype=float)

    for j in range(N):
            Jj = J_blocks[j]
            rows = slice(j*n, (j+1)*n)
            cols = slice(j*n, (j+1)*n)
            Jbig[rows, cols] = Jj

    return Jbig.tocsr()

def db_dz(z, Gamma, Gamma_pinv, n, N):
    # time samples
    x_tilde = Gamma @ z
    X = x_tilde_to_X(x_tilde, n, N)

    # ∂f/∂x in time is block diagonal, J_j = 2 sin(y_j) [[β_a, β_a],[β_b, β_b]]:
    # apply it block by block (vectorised, same result as Gamma_pinv @ (Jbig @ Gamma))
    s = np.sin(np.sqrt(2.0) * (X[0, :] + X[1, :]))
    base = np.array([[beta_a_p, beta_a_p],
                     [beta_b_p, beta_b_p]])
    G = Gamma.reshape(N, n, -1)                       # time-major: row j*n + a
    JG = np.einsum('j,ab,jbm->jam', 2.0 * s, base, G).reshape(n * N, -1)

    # chain rule
    return Gamma_pinv @ JG

def jacobian(z, L, Gamma, Gamma_pinv, n, N):
    return L + db_dz(z, Gamma, Gamma_pinv, n, N)
def newton_solve(z0, L, b_ext, Gamma, Gamma_pinv, n, N, tol=1e-10, maxit=50):
    z = z0.copy()
    for it in range(maxit):
        r = residual(z, L, b_ext, Gamma, Gamma_pinv, n, N)
        norm_r = np.linalg.norm(r)
        if norm_r < tol:
            print("Converged", it, norm_r)
            return z

        J = jacobian(z, L, Gamma, Gamma_pinv, n, N)
        dz = np.linalg.solve(J, -r)

        # optional damping
        z = z + dz
        print(it, norm_r, np.linalg.norm(dz))
    raise RuntimeError("Newton did not converge")

def powell_hybrid_solve(z0, L, b_ext, Gamma, Gamma_pinv, n, N,
                        tol=1e-10, maxit=50, use_jac=True, verbose = True):
    """
    Powell hybrid method (MINPACK hybr) via scipy.optimize.root.
    """

    def fun(z):
        return residual(z, L, b_ext, Gamma, Gamma_pinv, n, N)

    jac = (lambda z: jacobian(z, L, Gamma, Gamma_pinv, n, N)) if use_jac else None

    sol = root(
        fun, z0,
        jac=jac,
        method="hybr",
        tol=tol,
        options={"maxfev": maxit * (len(z0) + 1)}
    )

    if not sol.success:
        raise RuntimeError(f"Powell hybrid did not converge: {sol.message}")

    if verbose : print("Converged", sol.nfev, np.linalg.norm(sol.fun))
    return sol.x


In [ ]:
n   = 2 # number of modes
N_H = 4 # number of harmonics
# Number of instant for AFT
N = 900*N_H
# Uniform grid
omega = 2.0
nu = 2
tau_j = np.linspace(0, 100*np.pi, N)

z_0 = np.pad(z_0_init, (0,n*(2*N_H+1)-len(z_0_init)), constant_values=0.0)

M = np.array([[1.0, 0.0],
              [0.0, 0.25]])
C = np.array([[0.0, 0.0],
              [0.0, kappa_b_p]]) 
K = np.eye(2)
L = build_L(M, C, K, N_H, omega, nu)
Gamma = build_Gamma(tau_j, N_H, n, omega, nu)
Gamma_inverse = np.linalg.pinv(Gamma)

In [ ]:
b_ext = build_b_ext(N_H, eps_d_p, nu)
#z_sol = newton_solve(z_0, L, b_ext, Gamma, Gamma_inverse, n, N, maxit=50)
z_sol = powell_hybrid_solve(z_0, L, b_ext, Gamma, Gamma_inverse, n, N)


In [ ]:
z_sol

In [ ]:
def plot_phase_space(z_solution, N_H):
    """
    z_solution: array of length 2*(2*N_H+1)
    """

    # time grid (one period)
    t = np.linspace(0, 2*np.pi, 400)

    # split DOFs
    za = z_solution[::2]        # memory
    zb = z_solution[1::2]        # buffer

    # initialize
    x_a = np.zeros_like(t)
    p_a = np.zeros_like(t)
    x_b = np.zeros_like(t)
    p_b = np.zeros_like(t)

    # DC terms
    x_a += za[0] / np.sqrt(2)
    x_b += zb[0] / np.sqrt(2)

    # harmonics
    for k in range(1, N_H + 1):
        s_idx = 1 + 2*(k-1)
        c_idx = 2 + 2*(k-1)

        # memory (a)
        x_a += za[s_idx]*np.sin(k*t) + za[c_idx]*np.cos(k*t)
        p_a += k*za[s_idx]*np.cos(k*t) - k*za[c_idx]*np.sin(k*t)

        # buffer (b)
        x_b += zb[s_idx]*np.sin(k*t) + zb[c_idx]*np.cos(k*t)
        p_b += k/2*zb[s_idx]*np.cos(k*t) - k/2*zb[c_idx]*np.sin(k*t)

    a = (x_a + 1j*p_a)/np.sqrt(2)
    a_R = a*np.exp(1j*t)*1j
    import json 
    a_R_dict = [{"real": x.real, "imag": x.imag} for x in a_R.tolist()]
    #with open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/satellites_plus_12_02_2026_Hill.json", "w") as f:
    #    json.dump(a_R_dict, f)
        
    

    b = (x_b + 1j*p_b)/np.sqrt(2)
    b_R = b*np.exp(2j*t)
    b_R_dict = [{"real": x.real, "imag": x.imag} for x in b_R.tolist()]
    #with open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/satellites_buffer_12_02_2026_Hill.json", "w") as f:
    #    json.dump(b_R_dict, f)

    # ---- plots ----
    plt.figure()
    plt.plot(np.real(a_R), np.imag(a_R))
    plt.xlabel(r"$\Re(a_R)$")
    plt.ylabel(r"$\Im(a_R)$")
    plt.title("Phase portrait: memory (a)")
    plt.grid(True)
    plt.gca().set_aspect('equal')

    plt.figure()
    plt.plot(np.real(b_R), np.imag(b_R))
    plt.xlabel(r"$\Re(b_R)$")
    plt.ylabel(r"$\Im(b_R)$")
    plt.title("Phase portrait: buffer (b)")
    plt.grid(True)
    plt.gca().set_aspect('equal')
    plt.show()

In [ ]:
plot_phase_space(z_sol, N_H)

In [ ]:
# Satellite orbit at (eps_p, eps_d/g2) = (0.58, 8.12), read by limit_cycles.ipynb.
# Same frame as the plots above (a_R = i a e^{i tau}, b_R = b e^{2i tau}) and same format as the former satellites_*_12_02_2026_Hill.json.
# The second satellite is the image of the first under a -> -a (with tau -> tau + pi), i.e. -a_R; the buffer orbit is shared.
import json
OUT_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/"
t_orbit = np.linspace(0, 2*np.pi, 400)
za, zb = z_sol[::2], z_sol[1::2]
x_a = za[0]/np.sqrt(2) + sum(za[1+2*(k-1)]*np.sin(k*t_orbit) + za[2+2*(k-1)]*np.cos(k*t_orbit) for k in range(1, N_H+1))
p_a = sum(k*za[1+2*(k-1)]*np.cos(k*t_orbit) - k*za[2+2*(k-1)]*np.sin(k*t_orbit) for k in range(1, N_H+1))
x_b = zb[0]/np.sqrt(2) + sum(zb[1+2*(k-1)]*np.sin(k*t_orbit) + zb[2+2*(k-1)]*np.cos(k*t_orbit) for k in range(1, N_H+1))
p_b = sum(k/2*zb[1+2*(k-1)]*np.cos(k*t_orbit) - k/2*zb[2+2*(k-1)]*np.sin(k*t_orbit) for k in range(1, N_H+1))
a_R = (x_a + 1j*p_a)/np.sqrt(2)*np.exp(1j*t_orbit)*1j
b_R = (x_b + 1j*p_b)/np.sqrt(2)*np.exp(2j*t_orbit)
print("mean |a_R| =", np.mean(np.abs(a_R)), " mean |b_R| =", np.mean(np.abs(b_R)))
for name, orbit in (("plus", a_R), ("minus", -a_R), ("buffer", b_R)):
    with open(OUT_DIR + f"cat_{name}_12_10_2026_Hill.json", "w") as f:
        json.dump([{"real": x.real, "imag": x.imag} for x in orbit.tolist()], f)


In [ ]:
plot_phase_space(z_0, N_H)

In [ ]:
Re_a_list = np.linspace(-1,1,20)#[-0.7894736842105263]#[0.1, -0.4736842105263158, -0.6842105263157895] #
Im_a_list = np.linspace(-1,1,20)#[0.1578947368421053]#[0.1, 0.894736842105263, -1.0]#
n   = 2 # number of modes
N_H = 4 # number of harmonics
# Number of instant for AFT
N = 900*N_H
# Uniform grid
omega = 2.0
nu = 2
tau_j = np.linspace(0, 100*np.pi, N)

M = np.array([[1.0, 0.0],
              [0.0, 0.25]])
C = np.array([[0.0, 0.0],
              [0.0, kappa_b_p]]) 
K = np.eye(n)
L = build_L(M, C, K, N_H, omega, nu)
Gamma = build_Gamma(tau_j, N_H, n, omega, nu)
Gamma_inverse = np.linalg.pinv(Gamma)
b_ext = build_b_ext(N_H, eps_d_p, nu)
z_sol_list=[]
for re_a in Re_a_list:
    print(re_a)
    for im_a in Im_a_list:
        z_0_init = np.array([ 
            1.67375389e-01,  2.87830339e-01,
            re_a, im_a, 0.0,  0.0,
            0.0, 0.0, im_a,  5.69644528e-01,
            -9.22066305e-03, im_a, -1.18157915e-03, re_a,
            ])
        z_0 = np.pad(z_0_init, (0,n*(2*N_H+1)-len(z_0_init)), constant_values=0.0)
        
        try:
            #z_sol = newton_solve(
            #    z_0, L, b_ext, Gamma, Gamma_inverse,
            #    n, N, maxit=50
            #)
            z_sol = powell_hybrid_solve(z_0, L, b_ext, Gamma, Gamma_inverse, n, N, verbose = False)
            za = z_sol[::2]   # mode a
            za_1 = [round(coef, 2) for coef in za[1:3]]
            if za_1 == [ -0.,  0.] : 
                print(re_a, im_a)
            z_sol_list.append(z_sol)
        except Exception as e:
            print(e)
            pass

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax_a, ax_b = axes

for z_sol in z_sol_list:

    if z_sol is None:
        continue

    # one HB period
    t = np.linspace(0, 2*np.pi, 400)

    # split DOFs (time-major Fourier ordering)
    za = z_sol[::2]   # mode a
    zb = z_sol[1::2]  # mode b

    # initialize
    x_a = np.zeros_like(t)
    p_a = np.zeros_like(t)
    x_b = np.zeros_like(t)
    p_b = np.zeros_like(t)

    # DC
    x_a += za[0] / np.sqrt(2)
    x_b += zb[0] / np.sqrt(2)

    # harmonics
    for k in range(1, N_H + 1):
        s_idx = 1 + 2*(k-1)
        c_idx = 2 + 2*(k-1)

        # mode a
        x_a += za[s_idx]*np.sin(k*t) + za[c_idx]*np.cos(k*t)
        p_a += k*za[s_idx]*np.cos(k*t) - k*za[c_idx]*np.sin(k*t)

        # mode b  (remember physical frequency factor 1/2 in your model)
        x_b += zb[s_idx]*np.sin(k*t) + zb[c_idx]*np.cos(k*t)
        p_b += (k/2)*zb[s_idx]*np.cos(k*t) - (k/2)*zb[c_idx]*np.sin(k*t)

    # complex amplitudes
    a = (x_a + 1j*p_a)/np.sqrt(2)
    b = (x_b + 1j*p_b)/np.sqrt(2)

    a_R = a*np.exp(1j*t)
    b_R = b*np.exp(2j*t)

    ax_a.plot(np.real(a_R), np.imag(a_R))
    ax_b.plot(np.real(b_R), np.imag(b_R))

# formatting
ax_a.set_title("Phase portrait: mode a")
ax_a.set_xlabel(r"$\Re(a_R)$")
ax_a.set_ylabel(r"$\Im(a_R)$")
ax_a.set_xlim([-.5,.5])
ax_a.set_ylim([-.5,.5])
ax_a.set_aspect('equal')
ax_a.grid(True)

ax_b.set_title("Phase portrait: mode b")
ax_b.set_xlabel(r"$\Re(b_R)$")
ax_b.set_ylabel(r"$\Im(b_R)$")
ax_b.set_xlim([-.2,.5])
ax_b.set_ylim([-.2,.2])
ax_b.set_aspect('equal')
ax_b.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
za_1_list=[]
for z_sol in z_sol_list:
    if z_sol is None:
        continue
    za = z_sol[::2]   # mode a
    za_1 = [round(coef, 2) for coef in za[1:3]]
    za_1_list.append(za_1)

In [ ]:
1/np.sqrt(2)

In [ ]:
np.unique(np.array(za_1_list), axis = 0)[10:]

In [ ]:
za_1_list[1]==za_1_list[0]

In [ ]:
np.array(za_1_list)

# Checks

In [ ]:
import numpy as np

def build_Phi_terms(t, N_H):
    """Return Phi, dPhi, ddPhi for basis [1/sqrt2, sin, cos, ...]"""
    t = np.asarray(t)
    N = t.size
    Phi = np.zeros((N, 2*N_H+1))
    dPhi = np.zeros_like(Phi)
    ddPhi = np.zeros_like(Phi)

    Phi[:, 0] = 1/np.sqrt(2)

    for k in range(1, N_H+1):
        is_ = 1 + 2*(k-1)
        ic_ = 2 + 2*(k-1)

        Phi[:, is_] = np.sin(k*t)
        Phi[:, ic_] = np.cos(k*t)

        dPhi[:, is_] =  k*np.cos(k*t)
        dPhi[:, ic_] = -k*np.sin(k*t)

        ddPhi[:, is_] = -(k**2)*np.sin(k*t)
        ddPhi[:, ic_] = -(k**2)*np.cos(k*t)

    return Phi, dPhi, ddPhi

def build_Gamma_from_Phi(Phi, n):
    return np.kron(Phi, np.eye(n))

def check_L_vs_Gamma(L, t, N_H, n, M, C, K, trials=5, tol=1e-10):
    N = len(t)
    Phi, dPhi, ddPhi = build_Phi_terms(t, N_H)

    Gamma   = build_Gamma_from_Phi(Phi, n)
    Gamma_d = build_Gamma_from_Phi(dPhi, n)
    Gamma_dd= build_Gamma_from_Phi(ddPhi, n)

    # time-domain linear operator on stacked vectors
    Mbig = np.kron(np.eye(N), M)  # (nN x nN) but note stacking convention!
    Cbig = np.kron(np.eye(N), C)
    Kbig = np.kron(np.eye(N), K)

    # IMPORTANT: above assumes time-major stacking [x(t1); x(t2); ...]
    # If your tilde x is DOF-major [x1(t1..tN); x2(t1..tN)], use this instead:
    #Mbig = np.kron(M, np.eye(N))
    #Cbig = np.kron(C, np.eye(N))
    #Kbig = np.kron(K, np.eye(N))

    for r in range(trials):
        z = np.random.randn(n*(2*N_H+1))

        x_tilde   = Gamma   @ z
        dx_tilde  = Gamma_d @ z
        ddx_tilde = Gamma_dd@ z

        # time-domain linear result
        lin_time = Mbig @ ddx_tilde + Cbig @ dx_tilde + Kbig @ x_tilde

        # Fourier-domain linear result mapped back to time
        lin_freq_time = Gamma @ (L @ z)

        err = np.linalg.norm(lin_time - lin_freq_time) / max(1.0, np.linalg.norm(lin_time))
        print(f"trial {r}: relative error = {err:.3e}")

        if err > tol:
            print("❌ Inconsistent (likely ordering or sign/scaling mismatch).")
            return False

    print("✅ L is consistent with Gamma (within tolerance).")
    return True


In [ ]:
n = 2
N_H = 5
N = 16*N_H
t = np.linspace(0, 2*np.pi, N, endpoint=False)  # or your period

# put your matrices here
M = np.array([[1.0, 0.0],[0.0, 0.25]])
C = np.array([[0.0, 0.0],[0.0, kappa_b_p]])
K = np.eye(2)

L = build_L(M, C, K, N_H)  # make sure your build_L uses (M,C,K) in this order

check_L_vs_Gamma(L, t, N_H, n, M, C, K)


# Floquet coefficients

In [ ]:
from scipy.linalg import eig as scipy_eig

In [ ]:
def complex_harmonics(U, N_H, n):
    """
    Eigenvectors u (columns of U, ordering [c0, s1, c1, ...]) in the complex basis e^{ik tau}, k = -N_H..N_H:
        c_k cos(k t) + s_k sin(k t) = (c_k - i s_k)/2 e^{ikt} + (c_k + i s_k)/2 e^{-ikt}
    Returns an array of shape (2N_H+1, n, ncols), index N_H + k.
    """
    A = np.zeros((2*N_H + 1, n, U.shape[1]), dtype=complex)
    A[N_H] = U[:n, :] / np.sqrt(2)
    for k in range(1, N_H + 1):
        s = U[(2*k - 1)*n:(2*k)*n, :]
        c = U[(2*k)*n:(2*k + 1)*n, :]
        A[N_H + k] = (c - 1j*s) / 2
        A[N_H - k] = (c + 1j*s) / 2
    return A


def harmonic_centroid(A, N_H):
    """
    Mean harmonic index of each eigenvector. A spurious copy lambda + i m of a genuine exponent has its
    eigenvector shifted by -m harmonics, so the genuine ones are centred on k = 0 (Lazarus & Thomas 2010).
    """
    w = np.sum(np.abs(A)**2, axis=1)                 # (2N_H+1, ncols)
    k = np.arange(-N_H, N_H + 1)[:, None]
    return np.sum(k*w, axis=0) / np.sum(w, axis=0)


def shifted_overlap(a, b, m, N_H):
    """|<a shifted by -m harmonics, b>| / (|a| |b|) for single eigenvectors a, b of shape (2N_H+1, n)."""
    if m > 0:
        a_s, b_s = a[m:], b[:-m]
    elif m < 0:
        a_s, b_s = a[:m], b[-m:]
    else:
        a_s, b_s = a, b
    return np.abs(np.vdot(a_s, b_s)) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-300)


def select_genuine(eigvals, eigvecs, N_H, n, tol=1e-3, overlap_min=0.9):
    """
    Keep the 2n genuine Floquet exponents among the 2n(2N_H+1) Hill eigenvalues.

    Each Floquet multiplier appears in the Hill spectrum as a family lambda + i m (m integer), whose
    eigenvectors are shifted copies of each other in harmonic space. Candidates are ranked by
    |harmonic centroid| (then |Im|), and only one member per family is kept. This matters at period
    doubling: a multiplier on the negative real axis has two members lambda = sigma ± i/2 with the same
    |Im|, and Moore's |Im| criterion alone would pick both and drop another multiplier.

    The returned exponents are folded into the first zone |Im| <= 1/2 (Moore's convention, as in the
    paper); the multipliers exp(lambda T) are unchanged. The eigenvectors are those of the centred member.
    """
    U = eigvecs[eigvecs.shape[0]//2:, :]          # v = [lambda u ; u] -> bottom half is u
    A = complex_harmonics(U, N_H, n)
    centroid = harmonic_centroid(A, N_H)
    order = np.lexsort((np.abs(np.imag(eigvals)), np.abs(centroid)))
    chosen = []
    for j in order:
        lam = eigvals[j]
        if not np.isfinite(lam):
            continue
        same_family = False
        for c in chosen:
            d = lam - eigvals[c]
            m = int(np.round(np.imag(d)))
            if (m != 0 and abs(np.real(d)) < tol and abs(np.imag(d) - m) < tol
                    and shifted_overlap(A[:, :, c], A[:, :, j], m, N_H) > overlap_min):
                same_family = True
                break
        if not same_family:
            chosen.append(j)
        if len(chosen) == 2*n:
            break
    chosen = np.array(chosen)
    lam = eigvals[chosen]
    lam_folded = lam - 1j*np.round(np.imag(lam))
    # a real negative multiplier has members sigma ± i/2: always report it with Im = +1/2
    lam_folded = np.where(np.abs(np.imag(lam_folded) + 0.5) < 1e-6, lam_folded + 1j, lam_folded)
    return lam_folded, eigvecs[:, chosen]


def liouville_residual(eigvals, M, C):
    """
    Liouville/Abel: product of the Floquet multipliers = exp(-tr(M^-1 C) T), i.e. the real parts of
    the 2n genuine exponents must sum to -tr(M^-1 C). Returns the mismatch (should be ~0).
    """
    return np.sum(np.real(eigvals)) + np.trace(np.linalg.solve(M, C))


def Floq_coef(z_sol, L, M, C, N_H, n):

    Delta_1 = np.zeros([(2*N_H + 1)*n, (2*N_H + 1)*n])

    # Put C on every diagonal block (including k=0)
    for i in range(2*N_H + 1):
        Delta_1[i*n:(i+1)*n, i*n:(i+1)*n] = C

    # Put ±2k M between the (sin,cos) blocks for each k=1..N_H
    for k in range(1, N_H + 1):
        i = 2*k - 1  # sin index
        j = 2*k      # cos index

        # upper diagonal (sin row, cos column)
        Delta_1[i*n:(i+1)*n, j*n:(j+1)*n] = -2*k * M

        # lower diagonal (cos row, sin column)
        Delta_1[j*n:(j+1)*n, i*n:(i+1)*n] =  2*k * M


    Delta_2 = block_diag(*([M] * (2*N_H + 1)))
    h_z = jacobian(z_sol, L, Gamma, Gamma_inverse, n, N)

    B_1 = np.block([[Delta_1, h_z],[-np.eye((2*N_H + 1)*n),np.zeros([(2*N_H + 1)*n, (2*N_H + 1)*n])]])
    B_2 = -np.block([[Delta_2, np.zeros([(2*N_H + 1)*n, (2*N_H + 1)*n])],[np.zeros([(2*N_H + 1)*n, (2*N_H + 1)*n]), np.eye((2*N_H + 1)*n)]])
    eigvals, eigvecs = scipy_eig(B_1, B_2)

    # keep the 2n genuine Floquet exponents (Detroux et al. 2015, Sec. 3.1)
    return select_genuine(eigvals, eigvecs, N_H, n)

In [ ]:
from scipy.optimize import linear_sum_assignment

def match_eigs(prev_eigvals, prev_eigvecs, new_eigvals, new_eigvecs,
                alpha=1.0, beta=5.0, T=2*np.pi, n_dof=2):
    """
    Match new eigenpairs to previous ones using both eigenvalue proximity
    and eigenvector overlap (subspace continuity). Returns a permutation
    array 'perm' such that new_eigvals[perm] aligns with prev_eigvals.

    Both terms are insensitive to the i*m ambiguity of Floquet exponents, which otherwise makes the
    branches swap after a period doubling (a real negative multiplier sits at Im(lambda) = ±1/2):
      - proximity is measured between multipliers mu = exp(lambda T), not exponents;
      - overlap is maximised over harmonic shifts m = -1, 0, 1 (members of a family are shifted copies).
    """
    n = len(prev_eigvals)
    cost = np.zeros((n, n))

    n_rows = prev_eigvecs.shape[0] // 2               # v = [lambda u ; u] -> bottom half is u
    N_H_v = (n_rows // n_dof - 1) // 2
    A_prev = complex_harmonics(prev_eigvecs[n_rows:, :], N_H_v, n_dof)
    A_new  = complex_harmonics(new_eigvecs[n_rows:, :], N_H_v, n_dof)

    mu_prev, mu_new = np.exp(prev_eigvals*T), np.exp(new_eigvals*T)
    mu_scale = np.max(np.abs(mu_prev)) + 1e-12

    for i in range(n):
        for j in range(n):
            dval     = np.abs(mu_prev[i] - mu_new[j]) / mu_scale
            overlap  = max(shifted_overlap(A_prev[:, :, i], A_new[:, :, j], m, N_H_v) for m in (-1, 0, 1))  # in [0,1]
            dvec     = 1.0 - overlap
            cost[i, j] = alpha * dval + beta * dvec

    row_ind, perm = linear_sum_assignment(cost)
    # row_ind is just 0..n-1 in order since cost is square; perm gives new->prev mapping
    return perm

In [ ]:
z_guess = np.array([-2.09501523e-02, -3.60273363e-02,  3.26246220e-01, -3.28386996e-17,
       -6.78344780e-03,  1.43507376e-17, -1.72514305e-04, -3.13979476e-03,
       -4.23320965e-03, -7.46552651e-02, -8.01435361e-04, -8.82665625e-03,
        5.05426053e-05,  4.29993690e-04,  7.79954440e-06,  6.01698525e-05,
        1.00817384e-04,  8.67341068e-04]) #cat
N_H = 16
N = 900*N_H
# Uniform grid
omega = 2.0
nu = 2
tau_j = np.linspace(0, 100*np.pi, N)
z_guess = np.pad(z_guess, (0,n*(2*N_H+1)-len(z_guess)), constant_values=0.0)
eps_p_list = np.linspace(0.1, 0.785, 200)

# z_guess was converged with the former g2 (J1): continue it in eps_d up to 10.12 g2 at eps_p = 0.5
# (the direct continuation fails at eps_p = 0.41), then in eps_p down to the first eps_p of the sweep
varphi_a, varphi_b = np.sqrt(Z_a)/np.sqrt(2)/phi_quantum, np.sqrt(Z_b)/np.sqrt(2)/phi_quantum
M = np.array([[1.0, 0.0],
            [0.0, 0.25]])
C = np.array([[0.0, 0.0],
            [0.0, kappa_b / w_b]])
L = build_L(M, C, np.eye(2), N_H, omega, nu)
Gamma = build_Gamma(tau_j, N_H, n, omega, nu)
Gamma_inverse = np.linalg.pinv(Gamma)

def solve_at(z, eps_p, eps_d):
    global beta_a_p, beta_b_p, B      # used by f_nl_time / db_dz
    beta_a_p, beta_b_p = 2*E_J*np.sin(eps_p)*varphi_a**2/(w_a), 2*E_J*np.sin(eps_p)*varphi_b**2/(w_b)
    B = np.array([beta_a_p, beta_b_p])
    return powell_hybrid_solve(z, L, build_b_ext(N_H, eps_d*varphi_b/w_b, nu), Gamma, Gamma_inverse, n, N, verbose=False)

for eps_d_s in np.linspace(10.12*g2_J1_of(0.5), 10.12*g2_of(0.5), 25):
    z_guess = solve_at(z_guess, 0.5, eps_d_s)
for eps_p_s in np.linspace(0.5, eps_p_list[0], 10):
    z_guess = solve_at(z_guess, eps_p_s, 10.12*g2_of(eps_p_s))
eigvals_list = []
prev_eigvals = None  # track last successful eigenvalues for adiabatic sorting

for epsilon_p in eps_p_list:
    # --- Build parameters ---
    g2 = g2_of(epsilon_p)
    epsilon_d = 10.12 *g2#0.75*g2_max

    varphi_a, varphi_b = np.sqrt(Z_a)/np.sqrt(2)/phi_quantum, np.sqrt(Z_b)/np.sqrt(2)/phi_quantum
    beta_a_p, beta_b_p = 2*E_J*np.sin(epsilon_p)*varphi_a**2/(w_a), 2*E_J*np.sin(epsilon_p)*varphi_b**2/(w_b)
    B = np.array([beta_a_p, beta_b_p])  # shape (2,)
    eps_d_p = epsilon_d*varphi_b/w_b
    kappa_b_p = kappa_b / w_b

    M = np.array([[1.0, 0.0],
                [0.0, 0.25]])
    C = np.array([[0.0, 0.0],
                [0.0, kappa_b_p]]) 
    K = np.eye(2)
    L = build_L(M, C, K, N_H, omega, nu)
    Gamma = build_Gamma(tau_j, N_H, n, omega, nu)
    Gamma_inverse = np.linalg.pinv(Gamma)
    b_ext = build_b_ext(N_H, eps_d_p, nu)

    try:
        z_sol    = powell_hybrid_solve(z_guess, L, b_ext, Gamma, Gamma_inverse, n, N, verbose=False)
        new_eigvals, new_eigvecs = Floq_coef(z_sol, L, M, C, N_H, n)

        if prev_eigvals is None:
            sorted_eigvals = new_eigvals
            sorted_eigvecs = new_eigvecs
        else:
            perm = match_eigs(prev_eigvals, prev_eigvecs, new_eigvals, new_eigvecs, T=2*np.pi*nu/omega)
            sorted_eigvals = new_eigvals[perm]
            sorted_eigvecs = new_eigvecs[:, perm]

        eigvals_list.append(sorted_eigvals)
        prev_eigvals = sorted_eigvals
        prev_eigvecs = sorted_eigvecs
        z_guess      = z_sol

    except Exception as e:
        print(f"Failed at epsilon_p={epsilon_p:.4f}: {e}")
        # Don't update z_guess or prev_eigvals — keep last good state
        eigvals_list.append(np.full(2*n, np.nan + 1j * np.nan))

eigvals_array = np.array(eigvals_list)  # shape (n_steps, 2n)
print("max Liouville residual:", np.nanmax([abs(liouville_residual(ev, M, C)) for ev in eigvals_array]))

In [ ]:
T=2*np.pi/(omega/2) # Period corresponding to the memory frequency (half of the drive period)
Poincare_eig = np.exp(eigvals_array*T)

In [ ]:
np.exp(1j*T/2)

In [ ]:
Poincare_eig.T.shape

In [ ]:
plt.figure()
for i in range(2*n):
    plt.plot(eps_p_list, np.abs(eigvals_array[:, i]))
plt.xlabel("Pump amplitude "+r"$\varepsilon_p$")
plt.ylabel("Absolute value of Floquet\n exponents")
plt.show()

In [ ]:
plt.figure()
for i in range(2*n):
    plt.plot(eps_p_list, np.real(eigvals_array[:, i]))
plt.xlabel("Pump amplitude "+r"$\varepsilon_p$")
plt.ylabel("Real part of Floquet\n exponents")
plt.show()

In [ ]:
plt.figure()
for i in range(2*n):
    plt.plot(eps_p_list, np.imag(eigvals_array[:, i]))
plt.xlabel("Pump amplitude "+r"$\varepsilon_p$")
plt.ylabel("Imaginary part of Floquet\n exponents")
plt.show()

In [ ]:
g2_list = g2_of(eps_p_list)
plt.figure()
for i in range(2*n):
    plt.plot(g2_list/2/np.pi*1e3, np.real(Poincare_eig[:, i]))
plt.axhline([-1], ls="--", color="r")
plt.xlabel(r"$g_2$"+" (MHz)")
plt.ylabel("Real part of Floquet \n multipliers")
plt.tight_layout()
#plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/article/floquet_multiplier_before_period_doubling_08_10_2026.pdf")
plt.show()

In [ ]:
plt.figure()
for i in range(2*n):
    plt.plot(np.real(Poincare_eig[:, i]), np.imag(Poincare_eig[:, i]))

# Unit circle
theta = np.linspace(0, 2*np.pi, 100)
plt.plot(np.cos(theta), np.sin(theta), 'k-', lw=1, label='unit circle')

plt.axis('equal')
plt.xlabel("Real part of Floquet \n multipliers")
plt.ylabel("Imaginary part of Floquet \n multipliers")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Store what is plotted above (and the exponents), to replot in plot_floquet_multipliers.ipynb
import json
floquet_data = {
    "description": "Floquet exponents and multipliers of the period-1 (satellite) orbit vs eps_p, from Hill_method_corrected.ipynb",
    "eps_d_over_g2": 10.12,          # value used in the sweep cell
    "g2_definition": "E_J sin(eps_p) phi_a^2 phi_b",
    "kappa_b": kappa_b,
    "N_H": N_H,
    "T": T,
    "eps_p": eps_p_list.tolist(),
    "g2_MHz": (g2_list/2/np.pi*1e3).tolist(),
    "exponents_real": np.real(eigvals_array).tolist(),     # shape (len(eps_p), 2n), NaN where the solver failed
    "exponents_imag": np.imag(eigvals_array).tolist(),
    "multipliers_real": np.real(Poincare_eig).tolist(),
    "multipliers_imag": np.imag(Poincare_eig).tolist(),
}
with open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/chicago_poster_2026/floquet_cat_12_10_2026.json", "w") as f:
    json.dump(floquet_data, f)

In [ ]:
g2_list = g2_of(eps_p_list)
plt.figure()
for i in range(2*n):
    plt.plot(g2_list/2/np.pi*1e3, np.imag(Poincare_eig[:, i]))
plt.axhline([-1], ls="--", color="r")
plt.xlabel(r"$g_2$"+" (MHz)")
plt.ylabel("Imaginary part of Floquet \n multipliers")
plt.tight_layout()
#plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/poster_chicago/floquet_multiplier_before_period_doubling_08_10_2026.svg")
plt.show()